## Task Configuration

In [ ]:
TASK = "sentiment"   # "sentiment" or "hate"


## Imports

In [ ]:
import os, gc, re, random, zipfile, warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

from transformers import (
    AutoTokenizer, AutoModel,
    CLIPModel, CLIPProcessor,
    BlipProcessor, BlipForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  Task: {TASK}")


## Seed & Paths

In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)

BASE         = "/kaggle/input/sharedtask9"
CODEMIX_BASE = "/kaggle/input/datasets/arniedebnath/nepali-code-mixed/Code-Mixed"

TRAIN_CSV     = os.path.join(BASE, "train-20260216T180432Z-1-001/train/train.csv")
TRAIN_IMG_DIR = os.path.join(BASE, "train-20260216T180432Z-1-001/train/train_images")
LABEL_DIR     = os.path.join(BASE, "train-20260216T180432Z-1-001/train/train_images_labels")
VAL_CSV       = os.path.join(BASE, "val-20260216T180517Z-1-001/val/val.csv")
VAL_IMG_DIR   = os.path.join(BASE, "val-20260216T180517Z-1-001/val/val_no_labels/val_images")
TEST_CSV      = os.path.join(BASE, "test_release-20260216T180552Z-1-001/test_release/index_text_test.csv")
TEST_IMG_DIR  = os.path.join(BASE, "test_release-20260216T180552Z-1-001/test_release/test_images")
OUTPUT_DIR    = "/kaggle/working"


## Helper Functions

In [ ]:
def strip_ext(name):
    for ext in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
        if str(name).lower().endswith(ext): return str(name)[:-len(ext)]
    return str(name)

def find_img(img_dir, idx):
    idx = strip_ext(str(idx))
    for ext in ["", ".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(img_dir, idx + ext)
        if os.path.isfile(p): return p
    return None

def clean_text(t):
    if pd.isna(t): return ""
    return re.sub(r'\s+', ' ', str(t).strip())

def find_subdir(parent, keyword):
    if not os.path.exists(parent): return None
    matches = []
    for name in os.listdir(parent):
        if keyword.lower() in name.lower():
            full = os.path.join(parent, name)
            if os.path.isdir(full):
                n_files = len(os.listdir(full))
                matches.append((n_files, full))
    if not matches: return None
    return sorted(matches, reverse=True)[0][1]


## Label Mapping

In [ ]:
if os.path.exists(LABEL_DIR):
    train_df_tmp = pd.read_csv(TRAIN_CSV)
    folder_names = sorted([d for d in os.listdir(LABEL_DIR)
                           if os.path.isdir(os.path.join(LABEL_DIR, d))])
    folder_ids = {}
    for fn in folder_names:
        folder_ids[fn] = set(strip_ext(f)
                             for f in os.listdir(os.path.join(LABEL_DIR, fn)))
    int_to_name = {}
    for lab in sorted(train_df_tmp["label"].dropna().unique()):
        li      = int(lab)
        csv_ids = set(strip_ext(x) for x in
                      train_df_tmp[train_df_tmp["label"] == lab]["index"].astype(str).values)
        int_to_name[li] = max(folder_ids,
                              key=lambda fn: len(csv_ids & folder_ids[fn]))
    ID2LABEL = {int(k): str(v) for k, v in int_to_name.items()}
    del train_df_tmp
else:
    if TASK == "sentiment":
        ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}
    else:
        ID2LABEL = {0: "non-hate", 1: "hate"}

NUM_LABELS = len(ID2LABEL)
print(f"Label mapping: {ID2LABEL}")


## Load & Pool Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

for df in [train_df, val_df]:
    df["label_int"] = df["label"].apply(lambda x: int(x) if pd.notna(x) else -1)
train_df = train_df[train_df["label_int"].isin(ID2LABEL.keys())].reset_index(drop=True)
val_df   = val_df[val_df["label_int"].isin(ID2LABEL.keys())].reset_index(drop=True)

for df in [train_df, val_df, test_df]:
    df["clean_text"] = df["text"].apply(clean_text)

train_df["img_dir"] = TRAIN_IMG_DIR
val_df["img_dir"]   = VAL_IMG_DIR
test_df["img_dir"]  = TEST_IMG_DIR
train_df["source"]  = "mono"
val_df["source"]    = "mono"

pooled_df = pd.concat([train_df, val_df], ignore_index=True)
print(f"Pooled mono: {len(pooled_df)} samples ({len(train_df)} train + {len(val_df)} val)")
print(f"Test: {len(test_df)} samples")


## Build Code-Mixed Dataset

In [ ]:
def build_codemix_df(task="sentiment"):
    rows = []
    if task == "sentiment":
        sent_root = find_subdir(CODEMIX_BASE, "Sentiment Analysis")
        if sent_root is None:
            print(f"Sentiment folder not found in {CODEMIX_BASE}"); return None
        label_stems = [("negative", "neg", 0), ("neutral", "neu", 1), ("positive", "pos", 2)]
        for label_name, stem, label_int in label_stems:
            outer = find_subdir(sent_root, stem)
            if outer is None: print(f"Missing: {label_name} (stem={stem})"); continue
            inner = outer
            for sub in sorted(os.listdir(outer)):
                sp = os.path.join(outer, sub)
                if os.path.isdir(sp): inner = sp; break
            imgs = [f for f in os.listdir(inner)
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
            for fn in imgs:
                rows.append({"index": fn, "label_int": label_int,
                             "clean_text": "", "img_dir": inner, "source": "codemix"})
            print(f"  {label_name} (folder='{os.path.basename(inner)}'): {len(imgs)}")
    else:
        hate_root = find_subdir(CODEMIX_BASE, "hate speech")
        if hate_root is None:
            print(f"Hate folder not found in {CODEMIX_BASE}"); return None
        seen = set()
        for label_name, label_int in [("non hate", 0), ("non-hate", 0), ("hate", 1)]:
            outer = find_subdir(hate_root, label_name)
            if outer is None or outer in seen: continue
            seen.add(outer)
            inner = outer
            for sub in os.listdir(outer):
                sp = os.path.join(outer, sub)
                if os.path.isdir(sp): inner = sp; break
            imgs = [f for f in os.listdir(inner)
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
            for fn in imgs:
                rows.append({"index": fn, "label_int": label_int,
                             "clean_text": "", "img_dir": inner, "source": "codemix"})
            print(f"  label={label_int}: {len(imgs)}")
    if not rows: return None
    df = pd.DataFrame(rows).reset_index(drop=True)
    print(f"Total code-mixed: {len(df)}")
    return df

codemix_df = build_codemix_df(TASK)
if codemix_df is not None:
    print(f"Code-mixed label dist:\n{codemix_df['label_int'].value_counts().sort_index()}")


## OCR (EasyOCR)

In [ ]:
def run_ocr(df):
    try:
        import easyocr
        reader  = easyocr.Reader(['ne', 'en'], gpu=(DEVICE == 'cuda'))
        use_ocr = True
    except Exception as e:
        print(f"EasyOCR unavailable: {e}"); use_ocr = False
    texts = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="OCR"):
        if not use_ocr: texts.append(""); continue
        path = find_img(row["img_dir"], row["index"])
        try:
            r = reader.readtext(path, detail=0, paragraph=True) if path else []
            texts.append(" ".join(r).strip())
        except: texts.append("")
    if use_ocr: del reader; gc.collect()
    return texts

def combine_texts(csv_t, ocr_t):
    parts = [t.strip() for t in [csv_t, ocr_t] if t and t.strip()]
    if len(parts) == 2 and parts[0] == parts[1]: parts = [parts[0]]
    return " [SEP] ".join(parts)

pooled_df["ocr"]   = run_ocr(pooled_df)
test_df["ocr"]     = run_ocr(test_df)
pooled_df["ctext"] = [combine_texts(c, o) for c, o in zip(pooled_df["clean_text"], pooled_df["ocr"])]
test_df["ctext"]   = [combine_texts(c, o) for c, o in zip(test_df["clean_text"], test_df["ocr"])]

if codemix_df is not None:
    codemix_df["ocr"]   = run_ocr(codemix_df)
    codemix_df["ctext"] = [combine_texts(c, o) for c, o in zip(codemix_df["clean_text"], codemix_df["ocr"])]


## BLIP Captions

In [ ]:
def get_captions(df, bs=8):
    try:
        from transformers import Blip2Processor, Blip2ForConditionalGeneration
        proc  = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
        model = Blip2ForConditionalGeneration.from_pretrained(
            "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16).to(DEVICE)
        print("BLIP-2 loaded")
    except:
        proc  = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
        model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-large").to(DEVICE)
        print("BLIP-large loaded (fallback)")
    model.eval()
    caps = []; rows = list(df.itertuples())
    for i in tqdm(range(0, len(rows), bs), desc="Captions"):
        batch = rows[i:i+bs]
        imgs  = [Image.open(find_img(r.img_dir, r.index)).convert("RGB")
                 if find_img(r.img_dir, r.index)
                 else Image.new("RGB", (224, 224), (128, 128, 128))
                 for r in batch]
        inp = proc(images=imgs, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=60, num_beams=4)
        caps.extend(proc.batch_decode(out, skip_special_tokens=True))
    del model, proc; gc.collect(); torch.cuda.empty_cache()
    return caps

pooled_df["cap"] = get_captions(pooled_df)
test_df["cap"]   = get_captions(test_df)
if codemix_df is not None:
    codemix_df["cap"] = get_captions(codemix_df)


## CLIP Feature Extraction

In [ ]:
try:
    _ = CLIPModel.from_pretrained("openai/clip-vit-large-patch14"); del _
    CLIP_NAME = "openai/clip-vit-large-patch14"; CLIP_DIM = 768
    print("CLIP ViT-L/14 (768-d)")
except:
    CLIP_NAME = "openai/clip-vit-base-patch32"; CLIP_DIM = 512
    print("CLIP ViT-B/32 fallback")

def get_clip(df, bs=32):
    proc  = CLIPProcessor.from_pretrained(CLIP_NAME)
    model = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE); model.eval()
    ie_l, ce_l, s_l = [], [], []
    rows     = list(df.itertuples())
    captions = df["cap"].tolist()
    for i in tqdm(range(0, len(rows), bs), desc="CLIP"):
        batch = rows[i:i+bs]; bcap = captions[i:i+bs]
        imgs  = [Image.open(find_img(r.img_dir, r.index)).convert("RGB")
                 if find_img(r.img_dir, r.index)
                 else Image.new("RGB", (224, 224), (128, 128, 128))
                 for r in batch]
        inp = proc(text=bcap, images=imgs, return_tensors="pt",
                   padding=True, truncation=True, max_length=77).to(DEVICE)
        with torch.no_grad():
            out = model(**inp)
            ie  = F.normalize(out.image_embeds, dim=-1)
            te  = F.normalize(out.text_embeds,  dim=-1)
        cs = 2.5 * torch.clamp((ie * te).sum(-1), min=0)
        ie_l.append(ie.cpu().numpy())
        ce_l.append(te.cpu().numpy())
        s_l.append(cs.cpu().numpy())
    del model, proc; gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(ie_l), np.concatenate(ce_l), np.concatenate(s_l)

POOLED_IE, POOLED_CE, POOLED_CS = get_clip(pooled_df)
TEST_IE,   TEST_CE,   TEST_CS   = get_clip(test_df)

if codemix_df is not None:
    CM_IE, CM_CE, CM_CS = get_clip(codemix_df)
    print(f"CLIP done. Pooled:{POOLED_IE.shape}, Codemix:{CM_IE.shape}, Test:{TEST_IE.shape}")
else:
    CM_IE = CM_CE = CM_CS = None
    print(f"CLIP done. Pooled:{POOLED_IE.shape}, Test:{TEST_IE.shape}")


## Tokenizer, Class Weights & Loss Functions

In [ ]:
BACKBONE  = "xlm-roberta-large"
MAX_LEN   = 192
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

all_labels    = pooled_df["label_int"].values.astype(int)
class_weights = compute_class_weight("balanced",
                    classes=np.array(sorted(ID2LABEL.keys())),
                    y=all_labels)
CW = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"Class weights: {dict(enumerate(class_weights.round(3)))}")

class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__(); self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()

focal_loss = FocalLoss(alpha=CW, gamma=2.0)
ce_loss_fn = nn.CrossEntropyLoss(weight=CW, label_smoothing=0.05)


## MemeDataset

In [ ]:
class MemeDataset(Dataset):
    def __init__(self, texts, caps, img_embs, cap_embs, clip_s,
                 labels=None, is_train=True):
        self.texts    = texts
        self.caps     = caps
        self.img_embs = img_embs
        self.cap_embs = cap_embs
        self.clip_s   = clip_s
        self.labels   = labels
        self.is_train = is_train

    def __len__(self): return len(self.texts)

    def _aug(self, text):
        if not self.is_train or not text.strip(): return text
        words = text.split(); r = random.random()
        if r < 0.25:
            return " ".join(tokenizer.mask_token if random.random() < 0.15 else w
                            for w in words)
        elif r < 0.45:
            kept = [w for w in words if random.random() > 0.10]
            return " ".join(kept) if kept else text
        elif r < 0.60:
            for i in range(len(words) - 1):
                if random.random() < 0.10: words[i], words[i+1] = words[i+1], words[i]
            return " ".join(words)
        return text

    def __getitem__(self, idx):
        enc = tokenizer(self._aug(self.texts[idx]), self.caps[idx],
                        max_length=MAX_LEN, truncation="longest_first",
                        padding="max_length", return_tensors="pt")
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "img_emb": torch.tensor(self.img_embs[idx], dtype=torch.float32),
            "cap_emb": torch.tensor(self.cap_embs[idx], dtype=torch.float32),
            "clip_s":  torch.tensor(self.clip_s[idx],   dtype=torch.float32),
        }
        if self.labels is not None:
            item["label"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


## MemeModel

In [ ]:
class MemeModel(nn.Module):
    def __init__(self, clip_dim=CLIP_DIM, num_labels=NUM_LABELS,
                 hidden=256, dropout=0.4, freeze_n=18):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(BACKBONE)
        TEXT_DIM = self.backbone.config.hidden_size

        for p in self.backbone.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.backbone.encoder.layer):
            if i < freeze_n:
                for p in layer.parameters():
                    p.requires_grad = False

        self.q_proj = nn.Linear(TEXT_DIM, TEXT_DIM)
        self.k_proj = nn.Linear(clip_dim, TEXT_DIM)
        self.v_proj = nn.Linear(clip_dim, TEXT_DIM)
        self.cross  = nn.MultiheadAttention(TEXT_DIM, num_heads=8,
                                            dropout=0.1, batch_first=True)
        self.norm   = nn.LayerNorm(TEXT_DIM)

        self.img_proj = nn.Sequential(
            nn.Linear(clip_dim, hidden), nn.GELU(), nn.Dropout(dropout))
        self.cap_proj = nn.Sequential(
            nn.Linear(clip_dim, hidden), nn.GELU(), nn.Dropout(dropout))
        self.s_proj   = nn.Sequential(nn.Linear(1, 32), nn.GELU())

        in_dim = TEXT_DIM + hidden + hidden + 32
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim), nn.Dropout(dropout),
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, num_labels),
        )

    def forward(self, input_ids, attention_mask, img_emb, cap_emb, clip_s):
        out      = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_feat = out.last_hidden_state[:, 0, :]
        q = self.q_proj(cls_feat).unsqueeze(1)
        k = self.k_proj(img_emb).unsqueeze(1)
        v = self.v_proj(img_emb).unsqueeze(1)
        att, _ = self.cross(q, k, v)
        text_f = self.norm(cls_feat + att.squeeze(1))
        return self.head(torch.cat([
            text_f,
            self.img_proj(img_emb),
            self.cap_proj(cap_emb),
            self.s_proj(clip_s.unsqueeze(-1)),
        ], dim=-1))


## Training Config & train_fold

In [ ]:
BATCH        = 16
ACCUM        = 2
NUM_EPOCHS   = 20
PATIENCE     = 5
WARMUP_RATIO = 0.10
LR_BACKBONE  = 2e-5
LR_NEW       = 1e-4
N_FOLDS      = 4

def train_fold(fold_idx, tr_texts, tr_caps, tr_ie, tr_ce, tr_cs, tr_labels,
               va_texts, va_caps, va_ie, va_ce, va_cs, va_labels, seed=42):
    set_seed(seed)

    tr_ds = MemeDataset(tr_texts, tr_caps, tr_ie, tr_ce, tr_cs, tr_labels, is_train=True)
    va_ds = MemeDataset(va_texts, va_caps, va_ie, va_ce, va_cs, va_labels, is_train=False)
    tr_ld = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, num_workers=0)
    va_ld = DataLoader(va_ds, batch_size=BATCH * 2, shuffle=False, num_workers=0)

    model      = MemeModel(freeze_n=18).to(DEVICE)
    bb_params  = [p for p in model.backbone.parameters() if p.requires_grad]
    new_params = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    optimizer  = AdamW([{"params": bb_params, "lr": LR_BACKBONE},
                        {"params": new_params, "lr": LR_NEW}],
                       weight_decay=0.01, eps=1e-6)

    total_steps  = (len(tr_ld) // ACCUM) * NUM_EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler(enabled=(DEVICE == "cuda"))

    best_f1, best_ep, pat, best_state = 0., -1, 0, None

    for epoch in range(NUM_EPOCHS):
        model.train(); optimizer.zero_grad()
        t_preds, t_true = [], []
        pbar = tqdm(tr_ld, desc=f"[Fold{fold_idx+1}] Ep{epoch+1}/{NUM_EPOCHS}")

        for step, batch in enumerate(pbar):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            img  = batch["img_emb"].to(DEVICE)
            cap  = batch["cap_emb"].to(DEVICE)
            cs   = batch["clip_s"].to(DEVICE)
            lbls = batch["label"].to(DEVICE)

            with autocast(enabled=(DEVICE == "cuda")):
                logits = model(ids, mask, img, cap, cs)
                loss   = (0.7 * focal_loss(logits, lbls) +
                          0.3 * ce_loss_fn(logits, lbls)) / ACCUM

            scaler.scale(loss).backward()
            if (step + 1) % ACCUM == 0 or (step + 1) == len(tr_ld):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                scheduler.step(); optimizer.zero_grad()

            t_preds.extend(logits.argmax(-1).cpu().tolist())
            t_true.extend(lbls.cpu().tolist())
            pbar.set_postfix({"loss": f"{loss.item() * ACCUM:.4f}"})

        t_f1 = f1_score(t_true, t_preds, average="macro")

        model.eval(); v_preds, v_true = [], []
        with torch.no_grad():
            for batch in va_ld:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                img  = batch["img_emb"].to(DEVICE)
                cap  = batch["cap_emb"].to(DEVICE)
                cs   = batch["clip_s"].to(DEVICE)
                lbls = batch["label"].to(DEVICE)
                with autocast(enabled=(DEVICE == "cuda")):
                    logits = model(ids, mask, img, cap, cs)
                v_preds.extend(logits.argmax(-1).cpu().tolist())
                v_true.extend(lbls.cpu().tolist())

        v_f1  = f1_score(v_true, v_preds, average="macro")
        v_acc = accuracy_score(v_true, v_preds)
        print(f"[Fold{fold_idx+1}] Ep{epoch+1}: train_F1={t_f1:.4f} | val_F1={v_f1:.4f} val_acc={v_acc:.4f}")

        if v_f1 > best_f1:
            best_f1 = v_f1; best_ep = epoch + 1; pat = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            pat += 1
            if pat >= PATIENCE:
                print(f"Early stop ep{epoch+1}"); break

    model.load_state_dict(best_state)
    print(f"[Fold{fold_idx+1}] Best val F1={best_f1:.4f} at ep{best_ep}")
    return model, best_f1, v_preds, v_true


## K-Fold Cross Validation

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_models = []
fold_scores = []
oof_preds   = np.zeros(len(pooled_df), dtype=int)
oof_probs   = np.zeros((len(pooled_df), NUM_LABELS))

pooled_labels = pooled_df["label_int"].values.astype(int)
pooled_texts  = pooled_df["ctext"].tolist()
pooled_caps   = pooled_df["cap"].tolist()

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(pooled_texts, pooled_labels)):
    print(f"FOLD {fold_idx+1}/{N_FOLDS} — train:{len(tr_idx)}, val:{len(va_idx)}")

    tr_texts  = [pooled_texts[i] for i in tr_idx]
    tr_caps   = [pooled_caps[i]  for i in tr_idx]
    tr_ie     = POOLED_IE[tr_idx]
    tr_ce     = POOLED_CE[tr_idx]
    tr_cs     = POOLED_CS[tr_idx]
    tr_labels = [pooled_labels[i] for i in tr_idx]

    if codemix_df is not None:
        tr_texts  = tr_texts  + codemix_df["ctext"].tolist()
        tr_caps   = tr_caps   + codemix_df["cap"].tolist()
        tr_ie     = np.concatenate([tr_ie, CM_IE])
        tr_ce     = np.concatenate([tr_ce, CM_CE])
        tr_cs     = np.concatenate([tr_cs, CM_CS])
        tr_labels = tr_labels + codemix_df["label_int"].tolist()
        print(f"  Train: {len(tr_idx)} mono + {len(codemix_df)} codemix = {len(tr_labels)} total")

    va_texts  = [pooled_texts[i] for i in va_idx]
    va_caps   = [pooled_caps[i]  for i in va_idx]
    va_ie     = POOLED_IE[va_idx]
    va_ce     = POOLED_CE[va_idx]
    va_cs     = POOLED_CS[va_idx]
    va_labels = [pooled_labels[i] for i in va_idx]

    model, fold_f1, va_preds, va_true = train_fold(
        fold_idx,
        tr_texts, tr_caps, tr_ie, tr_ce, tr_cs, tr_labels,
        va_texts, va_caps, va_ie, va_ce, va_cs, va_labels,
        seed=42 + fold_idx,
    )

    fold_models.append(model)
    fold_scores.append(fold_f1)
    oof_preds[va_idx] = va_preds

    gc.collect(); torch.cuda.empty_cache()


## Out-of-Fold Evaluation

In [ ]:
target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
print(classification_report(pooled_labels, oof_preds, target_names=target_names, digits=4))
oof_f1 = f1_score(pooled_labels, oof_preds, average="macro")
print(f"Overall OOF Macro-F1: {oof_f1:.4f}")
print(f"Per-fold val F1s: {[round(s, 4) for s in fold_scores]}")
print(f"Mean fold F1: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")


## Inference Helper

In [ ]:
def infer(models, loader, scores):
    w = torch.tensor(scores, dtype=torch.float32)
    w = w / w.sum()
    all_p = []
    for m in models:
        m.eval(); run = []
        with torch.no_grad():
            for batch in loader:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                img  = batch["img_emb"].to(DEVICE)
                cap  = batch["cap_emb"].to(DEVICE)
                cs   = batch["clip_s"].to(DEVICE)
                with autocast(enabled=(DEVICE == "cuda")):
                    logits = m(ids, mask, img, cap, cs)
                run.append(logits.softmax(-1).cpu())
        all_p.append(torch.cat(run, dim=0))
    return sum(wi * p for wi, p in zip(w, all_p))


## Test Predictions & TTA

In [ ]:
te_ds = MemeDataset(
    test_df["ctext"].tolist(), test_df["cap"].tolist(),
    TEST_IE, TEST_CE, TEST_CS, labels=None, is_train=False,
)
te_ld = DataLoader(te_ds, batch_size=BATCH * 2, shuffle=False, num_workers=0)

clean_p = infer(fold_models, te_ld, fold_scores)

tta_list = []
for _ in range(5):
    aug_ds = MemeDataset(
        test_df["ctext"].tolist(), test_df["cap"].tolist(),
        TEST_IE, TEST_CE, TEST_CS, labels=None, is_train=True,
    )
    aug_ld = DataLoader(aug_ds, batch_size=BATCH * 2, shuffle=False, num_workers=0)
    tta_list.append(infer(fold_models, aug_ld, fold_scores))

final_p    = 0.5 * clean_p + 0.5 * torch.stack(tta_list).mean(0)
test_preds = final_p.argmax(-1).numpy()


## Save Predictions

In [ ]:
sub = pd.DataFrame({
    "index": test_df["index"].astype(str).values,
    "label": [int(p) for p in test_preds],
}).sort_values("index").reset_index(drop=True)

pred_csv = os.path.join(OUTPUT_DIR, "predictions.csv")
sub.to_csv(pred_csv, index=False)

zip_path = os.path.join(OUTPUT_DIR, "predictions.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_csv, "predictions.csv")

print(f"Task: {TASK}")
print(f"OOF Macro-F1: {oof_f1:.4f}")
print(f"Mean fold F1: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")
print(f"Per-fold F1s: {[round(s, 4) for s in fold_scores]}")
print(f"Submission: {zip_path}")
